In [ ]:
# 导入所需的库
import os
import numpy as np
import datetime
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks

In [ ]:
# GPU 显存优化设置
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Enabled memory growth for GPUs")
    except RuntimeError as e:
        print(f"Could not enable memory growth: {e}")

In [ ]:
# 参数和路径设置（用户可修改部分）
# 输入图像大小
IMG_SIZE = 256

# 数据路径设置
TRAIN_INPUT_DIR = 'F:/B208/DeepLearning/thrid_generation/gray_train/'
TRAIN_LABEL_DIR = 'F:/B208/DeepLearning/thrid_generation/binary_train/'
TEST_INPUT_DIR = 'F:/B208/DeepLearning/thrid_generation/gray_test/'
TEST_LABEL_DIR = 'F:/B208/DeepLearning/thrid_generation/binary_test/'

# 日志和模型保存路径
LOG_DIR = f"F:/B208/DeepLearning/tensorboard_logs/{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
MODEL_SAVE_PATH = 'F:/B208/DeepLearning/Model/best_binary_segmentation_model.h5'

In [ ]:
# 数据加载函数
def load_images(image_dir, target_size=(IMG_SIZE, IMG_SIZE)):
    images = []
    for filename in os.listdir(image_dir):
        img_path = os.path.join(image_dir, filename)
        try:
            img = tf.keras.preprocessing.image.load_img(img_path, color_mode='grayscale', target_size=target_size)
            img = tf.keras.preprocessing.image.img_to_array(img) / 255.0  # 归一化到 [0, 1]
            images.append(img)
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
    return np.array(images)

In [ ]:
# 加载数据
print("Loading training and testing data...")
train_images = load_images(TRAIN_INPUT_DIR)
train_labels = load_images(TRAIN_LABEL_DIR)
test_images = load_images(TEST_INPUT_DIR)
test_labels = load_images(TEST_LABEL_DIR)

# 划分验证集
train_images, val_images, train_labels, val_labels = train_test_split(
    train_images, train_labels, test_size=0.2, random_state=42
)
print("Data loaded successfully!")

In [ ]:
def build_unet_model(img_size):
    inputs = layers.Input(shape=(img_size, img_size, 1))

    # 编码器部分
    conv1 = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.LeakyReLU(alpha=0.1)(conv1)
    pool1 = layers.MaxPooling2D((2, 2))(conv1)

    conv2 = layers.Conv2D(64, (3, 3), padding='same')(pool1)
    conv2 = layers.BatchNormalization()(conv2)
    conv2 = layers.LeakyReLU(alpha=0.1)(conv2)
    pool2 = layers.MaxPooling2D((2, 2))(conv2)

    conv3 = layers.Conv2D(128, (3, 3), padding='same')(pool2)
    conv3 = layers.BatchNormalization()(conv3)
    conv3 = layers.LeakyReLU(alpha=0.1)(conv3)
    pool3 = layers.MaxPooling2D((2, 2))(conv3)

    conv4 = layers.Conv2D(256, (3, 3), padding='same')(pool3)
    conv4 = layers.BatchNormalization()(conv4)
    conv4 = layers.LeakyReLU(alpha=0.1)(conv4)
    pool4 = layers.MaxPooling2D((2, 2))(conv4)
    pool4 = layers.Dropout(0.3)(pool4)
    
    # 瓶颈层
    conv5 = layers.Conv2D(512, (3, 3), padding='same')(pool4)
    conv5 = layers.Conv2D(512, (3, 3), padding='same')(conv5)
    conv5 = layers.BatchNormalization()(conv5)
    conv5 = layers.ELU()(conv5)
    
    # 解码器部分 (反卷积 + 跳跃连接)
    upconv3 = layers.Conv2DTranspose(128, (3, 3), strides=(2, 2), padding='same')(conv4)
    upconv3 = layers.concatenate([upconv3, conv3])  # 跳跃连接
    upconv3 = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(upconv3)

    upconv2 = layers.Conv2DTranspose(64, (3, 3), strides=(2, 2), padding='same')(upconv3)
    upconv2 = layers.concatenate([upconv2, conv2])  # 跳跃连接
    upconv2 = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(upconv2)

    upconv1 = layers.Conv2DTranspose(32, (3, 3), strides=(2, 2), padding='same')(upconv2)
    upconv1 = layers.concatenate([upconv1, conv1])  # 跳跃连接
    upconv1 = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(upconv1)

    outputs = layers.Conv2D(1, (1, 1), padding='same', activation='sigmoid')(upconv1)

    model = models.Model(inputs, outputs)

    model.summary()
    return model

# 构建 U-Net 模型
model = build_unet_model(img_size)

In [ ]:
# 定义损失函数和评价指标
def l1_loss(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true - y_pred))

def mse_loss(y_true, y_pred):
    return tf.keras.losses.mean_squared_error(y_true, y_pred)

def ssim_loss(y_true, y_pred):
    return 1 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

def bce_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred)

def combined_loss(y_true, y_pred):
    return 0.55 * l1_loss(y_true, y_pred) + 0.15 * mse_loss(y_true, y_pred) + \
           0.15 * ssim_loss(y_true, y_pred) + 0.15 * bce_loss(y_true, y_pred)

def psnr_metric(y_true, y_pred):
    return tf.image.psnr(y_true, y_pred, max_val=1.0)

def ssim_metric(y_true, y_pred):
    return tf.image.ssim(y_true, y_pred, max_val=1.0)

In [ ]:
# 模型编译
model = build_unet_model(IMG_SIZE)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=[psnr_metric, ssim_metric]
)

In [ ]:
# 设置回调函数
tensorboard_callback = callbacks.TensorBoard(log_dir=LOG_DIR, histogram_freq=1)
lr_scheduler = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-8)
model_checkpoint = callbacks.ModelCheckpoint(
    MODEL_SAVE_PATH, monitor='val_loss', save_best_only=True, mode='min', verbose=1
)

In [80]:
# 训练模型
history = model.fit(
    train_images, train_labels,
    validation_data=(val_images, val_labels),
    epochs=200,
    batch_size=4,
    callbacks=[tensorboard_callback, lr_scheduler, model_checkpoint]
)

In [ ]:
# 保存最终模型
final_model_path = 'F:/B208/DeepLearning/Model/5_segmentation_model.h5'
model.save(final_model_path)
print(f"Model saved to {final_model_path}")

In [ ]:
# 输出 TensorBoard 指令
print(f"To visualize U-Net logs, run:\n    tensorboard --logdir={LOG_DIR}")